# Data Engineer (draft)

## 0. Thiết lập ban đầu

In [94]:
import sys
import os

# Go back 2 levels
project_root = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

# Add the project root to sys.path if it's not already there
if project_root not in sys.path:
    sys.path.append(project_root)

from src.common.config import (
    ROOT_DIR, DATA_DIR, 
    CLIP_FEATURES_DIR, MAP_KEYFRAMES_DIR, 
    OBJECTS_DIR, METADATA_DIR
)

## 1. Khám phá `clip_features/*.npy`

In [95]:
import numpy as np

sample = np.load(CLIP_FEATURES_DIR / "L21_V001.npy")
print("Sample shape: ", sample.shape)
print("Sample dtype: ", sample.dtype)
print(sample)

Sample shape:  (307, 512)
Sample dtype:  float16
[[ 0.02954    0.03067    0.03995   ...  0.06223   -0.05966   -0.000804 ]
 [ 0.002293  -0.02023    0.005737  ...  0.0961    -0.01501    0.0261   ]
 [-0.0003603 -0.04358   -0.01162   ...  0.1012    -0.00927    0.01402  ]
 ...
 [ 0.02052    0.01636   -0.03018   ...  0.0647    -0.001694   0.02588  ]
 [ 0.00462   -0.02487   -0.00134   ...  0.08093   -0.002817   0.01534  ]
 [-0.01648    0.0283    -0.01297   ...  0.06946   -0.02972   -0.02234  ]]


Video `L21_V001` có: 307 keyframe, mỗi dòng là 1 vector CLIP 512 chiều đã embedding (do model **CLIP ViT-B/32** sinh ra), đã được chuẩn hóa (L2-normalized $\rightarrow$ dùng **cosine similarity** khi so khớp với text query).

Khi nạp vào Qdrant thì vector phải là `float32`, nên cần convert lại trước khi upsert.

**$\Rightarrow$ Pipeline:** Câu truy vấn text $\rightarrow$ encode bằng **CLIP Text Encoder** $\rightarrow$ sinh ra vector text 512 chiều $\rightarrow$ vector text được so sánh với từng hàng (từng keyframe) trong mảng $\rightarrow$ keyframe có similarity cao nhất được coi là khớp nhất $\rightarrow$ trả về top-K kết quả cho Retrieval.

## 2. Khám phá `map_keyframes`

In [96]:
import pandas as pd
df_map = pd.read_csv(MAP_KEYFRAMES_DIR / "L21_V001.csv")
df_map

,n,pts_time,fps,frame_idx
0,1,0.0000,30.0,0
1,2,3.0000,30.0,90
2,3,8.7000,30.0,261
3,4,11.7333,30.0,351
4,5,13.7000,30.0,411
...,...,...,...,...
302,303,1240.7000,30.0,37221
303,304,1241.2300,30.0,37236
304,305,1245.3000,30.0,37359
305,306,1252.2300,30.0,37566


Ý nghĩa từng cột:
- `n`: số thứ tự keyframe (1-based), khớp với dòng `n-1` trong `clip_features/*.npy` và khớp với tên file trong `objects/`.
- `pts_time` (presentation timestamp): thời điểm (giây) mà keyframe này xuất hiện trong video. Bắt buộc phải có trong format nộp bài.
- `fps`: frame rate gốc của video, **không cố định giữa các video** (25 fps chiếm đa số với 781/873 video; còn lại là 30 fps, 29.97 fps và 26.44 fps).
- `frame_idx = round(pts_time × fps)`: số thứ tự frame tuyệt đối trong video gốc (chưa qua sampling), chính là `frame_id` **cần nạp vào Qdrant/ES**. BTC chấm điểm dựa trên frame gốc trong video, không phải thứ tự keyframe.

Nhận xét: Không phải sampling đều theo thời gian. Keyframe có thể đã được trích bằng **shot boundary detection / scene change detection** (chỉ giữ lại frame đại diện cho mỗi cảnh), không phải sampling ngẫu nhiên hay đều đặn.

**$\Rightarrow$ Pipeline:** Khi upsert vào Qdrant, mỗi record cần payload gồm `video_id`, `frame_id` / `pts_time` - lấy từ CSV này theo đúng chỉ số `n`.

## 3. Khám phá `objects/{video_id}/*.json`

In [97]:
import json
obj = json.load(open(OBJECTS_DIR / "L21_V001" / "001.json", encoding="utf-8"))
print("Object keys: ", obj.keys())

Object keys:  dict_keys(['detection_scores', 'detection_class_names', 'detection_class_entities', 'detection_boxes', 'detection_class_labels'])


In [98]:
print("Detection scores: ", obj["detection_scores"][:5])
print("Detection class names: ", obj["detection_class_names"][:5])
print("Detection class entities: ", obj["detection_class_entities"][:5])
print("Detection boxes: ", obj["detection_boxes"][:5])
print("Detection class labels: ", obj["detection_class_labels"][:5])

Detection scores:  ['0.79673874', '0.6866252', '0.6538629', '0.6242536', '0.5969766']
Detection class names:  ['/m/01jfsr', '/m/079cl', '/m/079cl', '/m/01jfsr', '/m/079cl']
Detection class entities:  ['Lantern', 'Skyscraper', 'Skyscraper', 'Lantern', 'Skyscraper']
Detection boxes:  [['0.46860334', '0.36642352', '0.63618606', '0.46714845'], ['0.40939528', '0.03066817', '0.6257758', '0.09503317'], ['0.4001567', '0.8219136', '0.6207597', '0.88224363'], ['0.3806457', '0.53154576', '0.7347561', '0.71973395'], ['0.3184604', '0.1778261', '0.5990897', '0.23009318']]
Detection class labels:  ['84', '379', '379', '84', '379']


`objects/{video_id}/{frame_id}.json` là output thô của model **Faster R-CNN** (format chuẩn TensorFlow Hub Object Detection - style Open Images), gồm kết quả detect object trên từng keyframe ảnh.

Ý nghĩa từng field:
- `detection_scores`: độ tin cậy (confidence level), sắp xếp giảm dần.
- `detection_class_names`: Machine ID của Open Images dataset dạng `/m/***`.
- `detection_class_entities`: tên nhãn dạng người đọc được (human-readable label) - nên dùng trực tiếp field này cho `objects_list` khi nạp vào Elasticsearch.
- `detection_boxes`: bounding box 4 số normalized [0, 1]: `[ymin, xmin, ymax, xmax]`.
- `detection_class_labels`: ID số nguyên nội bộ của class.

Nhận xét:
- Toàn bộ giá trị số lưu dưới dạng string, cần `float()`/`int()` trước khi xử lý.
- Confidence trải dài, nhiều detection là nhiễu $\rightarrow$ nên áp threshold trước khi đưa vào Elasticsearch.
- **Liên kết dữ liệu**: tên file `{n:03d}.json` khớp cột `n` trong `map_keyframes` $\rightarrow$ nối `clip_features[n-1]` (vector) - CSV dòng `n` (timestamp) - `objects/{video_id}/{n:03d}.json` (objects) khi build payload.

## 4. Khám phá `metadata/*.json`

In [99]:
meta = json.load(open(METADATA_DIR / "L21_V001.json", encoding="utf-8"))
print("Metadata keys: ", meta.keys())

Metadata keys:  dict_keys(['author', 'channel_id', 'channel_url', 'description', 'keywords', 'length', 'publish_date', 'thumbnail_url', 'title', 'watch_url'])


In [100]:
print("Title: ", meta.get("title", "N/A"))

Title:  60 Giây Sáng - Ngày 01082024 - HTV Tin Tức Mới Nhất 2024


In [101]:
print("Description: ", meta.get("description", "N/A"))

Description:  60 Giây Sáng - Ngày 01/08/2024 - HTV Tin Tức Mới Nhất 2024
► Đăng ký KÊNH để xem Tin Tức Mới Nhất: https://bit.ly/2HoUna4
Kênh Tin Tức Thời Sự 60 Giây Là Kênh Tổng Hợp Tin Tức - Sự Kiện - Giải trí Nhanh Nhất Việt Nam.

©️ Bản quyền Tin Tức thuộc HTV - Đài Truyền hình TP.HCM
©️ The copyright of this video belongs to HTV - Ho Chi Minh City Television
#TintucthoisuVietnam
#HTVTintuc #HTVnews #HTV60s #60Giay #60s
------------
Xem TV online tại Truyền hình HTVC:
✅ Web / Wap mobile :
► https://hplus.com.vn
► https://htvc.com.vn/
► https://htvc.tv/
✅ App mobile/TV HTVC :
► IOS: https://goo.gl/Dih3DB
► Android: https://goo.gl/XGBzxg
► Smart TV /STB : Tải HTVC for Android TV
.
✅ Theo dõi HTVC trên Mạng xã hội :
- Facebook Show : https://www.facebook.com/HPlusOfficial
- Instagram Show : https://www.instagram.com/htvc_truyen_hinh_internet/
- Lotus Show : https://lotus.vn/w/profile/18444255300272761.htm
- Tiktok News : https://www.tiktok.com/@htvc.tintuc
- Tiktok Phim : https://www.t

Ý nghĩa một số field:
- `description`: thường lẫn nhiều text quảng cáo/link mạng xã hội không liên quan nội dung video.
- `keywords`: tag do người upload tự nhập (SEO tags), **không phải AI-generated**.
- `length`: đơn vị giây (không phải `mm:ss`).
- `publish_date`: string dạng `DD/MM/YYYY` (không phải ISO), cần parse thủ công nếu muốn dùng.

Nhận xét:
- **Granularity khác 3 phần trước**: metadata là 1-file-per-video (không theo keyframe) $\rightarrow$ khi build payload sẽ lặp lại cho mọi keyframe cùng video; `title` + `description` là nguồn chính cho **Sparse Search (BM25)** trên Elasticsearch, kết hợp cùng `detection_class_entities`.
- `keywords` nhiễu, nhiều biến thể trùng lặp (có/không dấu, hoa/thường) và lặp giữa các video cùng kênh $\rightarrow$ nên hạ trọng số hoặc loại khỏi index.

## 5. Verify tính nhất quán toàn dataset (bắt buộc trước khi code ingestion thật)

`clip_features`, `map_keyframes`, `objects`, `metadata` đến từ các script trích xuất độc lập của BTC - nếu 1 trong số đó lệch dòng/thiếu file, việc ghép `(vector, frame_id, objects_list, metadata_text)` theo chỉ số `n` / `video_id` sẽ sai một cách âm thầm, không văng lỗi. Cần verify toàn bộ 873 video trước khi tin tưởng ghép 1-1.

In [102]:
# Scan whole dataset for CSV/.npy row count mismatches
mismatches = []
video_ids = sorted(p.stem for p in MAP_KEYFRAMES_DIR.glob("*.csv"))

for video_id in video_ids:
    csv_path = MAP_KEYFRAMES_DIR / f"{video_id}.csv"
    npy_path = CLIP_FEATURES_DIR / f"{video_id}.npy"

    if not npy_path.exists():
        mismatches.append((video_id, "missing .npy"))
        continue

    n_rows_csv = len(pd.read_csv(csv_path))
    n_rows_npy = np.load(npy_path).shape[0]

    if n_rows_csv != n_rows_npy:
        mismatches.append((video_id, f"csv={n_rows_csv} vs npy={n_rows_npy}"))

print(f"Checked {len(video_ids)} videos.")
print(f"Mismatches found: {len(mismatches)}")
print(mismatches[:20])

Checked 873 videos.
Mismatches found: 0
[]


In [103]:
# Scan whole dataset for CSV/objects file-count mismatches
mismatches_obj = []

for video_id in video_ids:
    csv_path = MAP_KEYFRAMES_DIR / f"{video_id}.csv"
    obj_dir = OBJECTS_DIR / video_id

    if not obj_dir.exists():
        mismatches_obj.append((video_id, "missing objects folder"))
        continue

    n_rows_csv = len(pd.read_csv(csv_path))
    n_files_obj = len(list(obj_dir.glob("*.json")))

    if n_rows_csv != n_files_obj:
        mismatches_obj.append((video_id, f"csv={n_rows_csv} vs objects={n_files_obj}"))

print(f"Checked {len(video_ids)} videos.")
print(f"Mismatches found: {len(mismatches_obj)}")
print(mismatches_obj[:20])

Checked 873 videos.
Mismatches found: 0
[]


In [104]:
# Check max keyframes per video (objects filename uses {n:03d}.json -> unsafe if n > 999)
max_n = max(len(pd.read_csv(MAP_KEYFRAMES_DIR / f"{vid}.csv")) for vid in video_ids)
print("Max keyframes per video:", max_n)
print("3-digit zero-padding is safe (max_n < 1000):", max_n < 1000)

Max keyframes per video: 632
3-digit zero-padding is safe (max_n < 1000): True


In [105]:
# Check metadata/*.json exists for every video_id
missing_meta = [vid for vid in video_ids if not (METADATA_DIR / f"{vid}.json").exists()]
print(f"Missing metadata: {len(missing_meta)}")
print(missing_meta[:20])

Missing metadata: 0
[]


In [106]:
# Sanity check: no keyframe timestamp should exceed the video's reported length
bad_length = []

for video_id in video_ids:
    max_pts = pd.read_csv(MAP_KEYFRAMES_DIR / f"{video_id}.csv")["pts_time"].max()
    length = json.load(open(METADATA_DIR / f"{video_id}.json", encoding="utf-8"))["length"]

    if max_pts > length:
        bad_length.append((video_id, max_pts, length))

print(f"Checked {len(video_ids)} videos.")
print(f"Videos where max(pts_time) > length: {len(bad_length)}")
print(bad_length[:20])

Checked 873 videos.
Videos where max(pts_time) > length: 33
[('L24_V012', np.float64(97.04), 97), ('L24_V044', np.float64(35.1766), 35), ('L25_V003', np.float64(1162.12), 1162), ('L25_V013', np.float64(1192.02), 1192), ('L25_V014', np.float64(1591.12), 1591), ('L25_V016', np.float64(1592.24), 1592), ('L25_V018', np.float64(1662.24), 1662), ('L25_V023', np.float64(1478.2), 1478), ('L25_V026', np.float64(1650.04), 1650), ('L25_V033', np.float64(1929.08), 1929), ('L25_V036', np.float64(1806.28), 1806), ('L25_V037', np.float64(879.24), 879), ('L25_V041', np.float64(1428.16), 1428), ('L25_V042', np.float64(1929.08), 1929), ('L25_V043', np.float64(1695.16), 1695), ('L25_V045', np.float64(1381.12), 1381), ('L25_V046', np.float64(1006.32), 1006), ('L25_V051', np.float64(2017.08), 2017), ('L25_V054', np.float64(2518.04), 2518), ('L25_V055', np.float64(1042.08), 1042)]


Kết quả: 33/873 video có `max(pts_time) > length`, nhưng chênh lệch đều dưới 1 giây (vd `97.04` vs `97`) - do `length` là số nguyên giây đã làm tròn, còn `pts_time` là float chính xác hơn. Không phải lỗi cấu trúc dữ liệu.

In [107]:
# Is (video_id, frame_idx) unique? -> decides what the DB record id can be keyed on
dup_videos = []
total_rows = total_unique = 0

for video_id in video_ids:
    fidx = pd.read_csv(MAP_KEYFRAMES_DIR / f"{video_id}.csv")["frame_idx"].astype(int).tolist()
    total_rows += len(fidx)
    total_unique += len(set(fidx))
    if len(set(fidx)) != len(fidx):
        dup_videos.append((video_id, len(fidx) - len(set(fidx))))

print(f"Videos with duplicate frame_idx: {len(dup_videos)}")
print(f"Total keyframes: {total_rows} | unique (video_id, frame_idx): {total_unique}")
print(f"=> records lost if id keyed on frame_id: {total_rows - total_unique}")
print("Worst offenders:", sorted(dup_videos, key=lambda x: -x[1])[:5])

Videos with duplicate frame_idx: 192
Total keyframes: 177321 | unique (video_id, frame_idx): 176707
=> records lost if id keyed on frame_id: 614
Worst offenders: [('L28_V006', 15), ('L25_V082', 12), ('L25_V086', 10), ('L26_V209', 10), ('L27_V011', 10)]


In [108]:
# Is n a safe id key, and does n order == temporal order? (needed for TRAKE)
bad_n, bad_pts = [], []

for video_id in video_ids:
    df = pd.read_csv(MAP_KEYFRAMES_DIR / f"{video_id}.csv")
    n = df["n"].astype(int).tolist()
    pts = df["pts_time"].astype(float).tolist()

    if n != list(range(1, len(n) + 1)):
        bad_n.append(video_id)
    if any(pts[i] >= pts[i + 1] for i in range(len(pts) - 1)):
        bad_pts.append(video_id)

print(f"Checked {len(video_ids)} videos.")
print(f"Videos where n is not exactly 1..N   : {len(bad_n)} {bad_n[:5]}")
print(f"Videos where pts_time not increasing : {len(bad_pts)} {bad_pts[:5]}")

Checked 873 videos.
Videos where n is not exactly 1..N   : 0 []
Videos where pts_time not increasing : 0 []


Kết quả (quan trọng nhất của mục 5):

- **`frame_idx` KHÔNG duy nhất trong 1 video**: 192/873 video có trùng, tổng cộng 614/177,321 keyframe. Nguyên nhân: `frame_idx = round(pts_time × fps)`, hai keyframe cách nhau dưới 1 frame (vd 19.20s và 19.24s ở 25 fps) làm tròn về cùng một giá trị. $\rightarrow$ **Không được dùng `(video_id, frame_id)` làm id của record** trong Qdrant/ES, nếu không 614 bản ghi bị ghi đè âm thầm mà `count()` vẫn báo thành công.
- **`n` an toàn để làm id**: đúng `1..N` ở 100% video.
- **Thứ tự `n` = thứ tự thời gian**: `pts_time` tăng nghiêm ngặt ở 100% video $\rightarrow$ ràng buộc thời gian của TRAKE dịch thẳng thành `n_1 < n_2 < ... < n_N`, so sánh số nguyên trên `n` thay vì so sánh chuỗi trên `frame_id`. Lưu ý: `n` chỉ so sánh được **trong cùng 1 video** (mọi video đều có `n = 1`).

$\Rightarrow$ **Chốt thiết kế:** id của record = `{batch}_{video_id}_{n}`; `frame_id` giữ kiểu `str` đúng spec, chỉ dùng để định danh và nộp bài, không dùng để so sánh thứ tự.